# The ₹0 AI Engineer — Live Lab Notebook
### Building Real Things Before You Graduate

This single notebook powers **two** workshop demos, end to end, on a **free Google account** — no GPU, no credit card, no local setup.

| Part | Module | What you'll do |
|------|--------|----------------|
| **A** | Module 2 — *Prompt Engineering as a Professional Discipline* | Run a **naive** prompt vs an **engineered** prompt on the same task, and **measure the token difference live** |
| **B** | Module 3 — *Building Real Applications Without a GPU* | Send a résumé to a real LLM API and get back **structured JSON** your code can use — reading the live `usage` numbers |

> **Run order:** top to bottom. Each cell builds on the one above. Press **▶** on each cell (or `Shift+Enter`).

---
*Instructor: Pramit Mitra · Principal Engineer, Data Science Platform — Dell Technologies*


## 🔧 Setup — do this once (≈ 60 seconds)

We use Google's **free** Gemini API. You need one free key from Google AI Studio.


### Step 1 · Install the SDK
The modern package is **`google-genai`** (the old `google-generativeai` is deprecated — ignore any tutorial that uses it).

In [ ]:
# Install the official Google Gen AI SDK (quiet)
!pip install -q -U google-genai
print("✅ SDK installed. Restart NOT required — continue to the next cell.")

: 

### Step 2 · Get your free API key

1. Open **https://aistudio.google.com/apikey** in a new tab
2. Sign in with any Google account
3. Click **Create API key** → copy it (it starts with `AIza...`)
4. **No credit card. No billing.** This is the free tier.

### Step 3 · Store the key safely in Colab (recommended)

Don't paste your key into a code cell where everyone can see it. Instead:

1. Click the **🔑 key icon** in the left sidebar of Colab (*Secrets*)
2. Click **+ Add new secret**
3. Name it exactly **`GEMINI_API_KEY`** and paste your key as the value
4. Toggle **Notebook access** ON

Then run the cell below. *(If you skip Secrets, the cell will fall back to asking you to paste the key privately.)*

In [ ]:
# Load the API key from Colab Secrets, with a safe fallback
#import os

api_key = None
try:
    from google.colab import userdata           # only exists inside Colab
    api_key = userdata.get("GeminiAPIKey")
    print("✅ Key loaded from Colab Secrets.")
except Exception:
    pass

if not api_key:
    # Fallback: prompt privately (input is hidden)
    import getpass
    api_key = getpass.getpass("GeminiAPIKey(hidden): ").strip()
    print("✅ Key captured for this session.")

assert api_key and api_key.startswith("AQ"), "That doesn't look like a Gemini key (should start with 'AIza')."


✅ Key captured for this session.


### Step 4 · Create the client and pick a free model

We default to a **free-tier Flash** model — fast and $0 to run. Model names change over time; if one is unavailable, the next cell lists what your key can actually use, and you can swap `MODEL` to any of them.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

# A fast, free-tier model. If this name ever errors, run the "list models"
# cell below and copy any *-flash model name here.
MODEL = "gemini-2.5-flash"

# Smoke test: one tiny call to confirm everything works
resp = client.models.generate_content(model=MODEL, contents="Reply with exactly: ready")
print("Model says:", resp.text.strip())
print("✅ You're connected. MODEL =", MODEL)

ModuleNotFoundError: No module named 'google'

<details>
<summary>🛟 <b>Troubleshooting / optional:</b> list the models your key can use</summary>

Run this only if the smoke test above failed with a model error.
</details>

In [ ]:
# OPTIONAL — only if you hit a model-not-found error above.
for m in client.models.list():
    name = m.name.split("/")[-1]
    if "flash" in name:          # show the cheap/free-friendly ones
        print(name)

### A tiny helper — our "token meter" 🪙

This wraps one API call and prints the **live usage numbers** returned by the model: input tokens, output tokens, and total. This meter is the instrument for the whole lab — watch it after every run.

In [ ]:
def run(prompt=None, *, system=None, user=None, schema=None, label=""):
    """Call the model once and print the live token meter.
    Pass either `prompt=...` (simple) OR `system=...` + `user=...` (pro)."""
    # Build the request
    if user is not None:
        contents = user
        cfg = types.GenerateContentConfig(system_instruction=system) if system else None
    else:
        contents = prompt
        cfg = None

    # If a JSON schema is requested, ask for structured output
    if schema is not None:
        cfg = cfg or types.GenerateContentConfig()
        cfg.response_mime_type = "application/json"
        cfg.response_schema = schema

    response = client.models.generate_content(model=MODEL, contents=contents, config=cfg)

    u = response.usage_metadata
    bar = "─" * 44
    print(bar)
    if label:
        print(f"🏷  {label}")
    print(f"🪙  input tokens : {u.prompt_token_count}")
    print(f"🪙  output tokens: {u.candidates_token_count}")
    print(f"🪙  TOTAL tokens : {u.total_token_count}")
    print(bar)
    print(response.text.strip())
    return response

print("✅ Helper ready. Use run(...) below.")

---
# 🅰️  PART A — Module 2: Naive vs. Engineered Prompt

**The experiment:** same task (*extract the skills a job wants*), two different prompts.
**The hypothesis:** the engineered prompt is **more reliable AND uses fewer tokens**.

We keep the **same input text** for both rounds so the comparison is fair.

In [ ]:
# The shared input — the SAME job description feeds both rounds.
JOB_DESCRIPTION = """
We're hiring a Data Engineer. You'll build batch and streaming pipelines on our
cloud lakehouse. Must have strong Python and SQL, hands-on Apache Spark, and
experience with a cloud platform (AWS or Azure). Nice to have: Kubernetes,
Airflow, and exposure to Iceberg or Delta Lake. You'll work closely with data
scientists to productionise ML features.
"""
print("Input ready —", len(JOB_DESCRIPTION.split()), "words")

### Round 1 · The **naive** prompt
What most people type the first time: polite, chatty, no structure. Run it **two or three times** and notice the answer's *shape changes each time*.

In [ ]:
naive_prompt = f"""Hey! Can you please take a look at this job description and kindly
tell me what skills they are looking for? Thank you so much!

{JOB_DESCRIPTION}"""

_ = run(prompt=naive_prompt, label="ROUND 1 — naive prompt")

👀 **Look at the meter and the output.** Typical problems you'll see:
- A chatty **preamble** (*"Sure! Here are the skills…"*) your code would have to strip
- The **format changes** between runs — sometimes a paragraph, sometimes bullets
- Wasted tokens on *"please / kindly / thank you"*

It works as a *chat*. It fails as a *feature*.

### Round 2 · The **engineered** prompt
Same task, redesigned using the anatomy from the slides: a **system role**, an **explicit JSON contract**, and **zero filler**. The user slot holds *only the data*.

In [ ]:
system_instruction = (
    "You extract skills from job descriptions. "
    "Return ONLY JSON with two keys: 'required' and 'preferred', "
    "each a list of short skill strings. No prose, no markdown."
)

_ = run(system=system_instruction, user=JOB_DESCRIPTION,
        label="ROUND 2 — engineered prompt")

✅ **Look at the meter again.** You should see:
- **Fewer total tokens** than Round 1 (we deleted the filler)
- The **same JSON shape every run** — predictable, parseable
- **No preamble** to strip

This is the whole point of Module 2: prompt design is a *measurable* discipline.

### The measured result — side by side
Let's count tokens for **both prompts** directly (without even generating), so the difference is undeniable.

In [ ]:
naive_tokens = client.models.count_tokens(model=MODEL, contents=naive_prompt).total_tokens

# For the engineered version, the model sees system + user together
engineered_contents = system_instruction + "\n\n" + JOB_DESCRIPTION
eng_tokens = client.models.count_tokens(model=MODEL, contents=engineered_contents).total_tokens

saved = naive_tokens - eng_tokens
pct = (saved / naive_tokens * 100) if naive_tokens else 0

print(f"Naive prompt input ......... {naive_tokens} tokens")
print(f"Engineered prompt input .... {eng_tokens} tokens")
print(f"Saved ...................... {saved} tokens  ({pct:.0f}% fewer)")
print()
print("Now imagine 50,000 calls/month:")
print(f"  {saved:,} tokens/call  ×  50,000  =  {saved*50000:,} fewer input tokens / month")

> 💡 **The Module 1 → Module 2 bridge in one line:** a prompt that saves \~100 tokens per call saves *millions* of tokens a month at scale. **Prompt design is cost engineering.**

*(Your exact numbers will differ from the slides — that's expected. The **direction** is always the same: structured + concise wins.)*

---
# 🅱️  PART B — Module 3: A Real App With No GPU

Now we treat the model as an **application component**, not a chat toy. We'll send a
résumé in and get back **guaranteed-shape JSON** — the kind your code can drop straight
into a database, a dashboard, or the next step of a pipeline.

The key upgrade: a **response schema**. We don't *ask nicely* for JSON — we *enforce* it.

In [ ]:
# A sample résumé (paste your own later to see it work on real data!)
RESUME = """
Aarti Sharma — Bengaluru, India
Software engineer with 4 years building data-heavy backend systems.
Skills: Python, FastAPI, PostgreSQL, Apache Spark, Docker, basic Kubernetes.
Experience:
 - Built an ETL pipeline processing 2TB/day at a fintech startup (2 years)
 - Developed REST APIs serving 10M requests/day
 - Migrated a legacy reporting system to a cloud lakehouse
Education: B.Tech, Computer Science, 2021.
Certifications: Databricks Lakehouse Fundamentals.
"""
print("Résumé ready —", len(RESUME.split()), "words")

### Define the JSON contract with a schema
We describe the **exact shape** we want back. The model is now *required* to fill this structure — no preamble, no markdown, no surprises.

In [ ]:
from pydantic import BaseModel
from typing import List

class Candidate(BaseModel):
    name: str
    years_experience: float
    skills: List[str]
    summary: str          # one-line recruiter summary
    seniority: str        # e.g. "junior", "mid", "senior"

resume_system = (
    "You are a precise résumé parser. Extract the requested fields accurately. "
    "If a field is unknown, make your best estimate from the text."
)

response = run(
    system=resume_system,
    user=RESUME,
    schema=Candidate,            # <-- enforces the JSON shape
    label="RÉSUMÉ → STRUCTURED JSON",
)

### Prove it's real data, not just text
Because the output is guaranteed JSON matching our schema, we can parse it and use the
fields **like any normal Python object** — `json.loads()` and done.

In [ ]:
import json

data = json.loads(response.text)     # guaranteed to parse — that's the power of a schema

print("Parsed object:\n")
print("  Name        :", data["name"])
print("  Experience  :", data["years_experience"], "years")
print("  Seniority   :", data["seniority"])
print("  Skills      :", ", ".join(data["skills"]))
print("  Summary     :", data["summary"])
print()
print("👉 This dict could now go straight into a database row, a Pandas frame,")
print("   a ranking function, or the next step of an agent. No string-wrangling.")

### 🧪 Your turn — change something and re-run

Try any of these, then re-run the two cells above and watch the `usage` meter move:

1. **Paste your own CV** into the `RESUME` string and see your real skills extracted.
2. **Add a field** to the `Candidate` schema — e.g. `top_company: str` or `languages: List[str]` — and watch the model fill it.
3. **Switch models:** set `MODEL = "gemini-2.5-flash-lite"` for an even cheaper tier and compare token counts.

> Every change is free on the free tier. This is the whole thesis of the workshop:
> **you don't need a GPU — you need judgement about *which* API to call and *how* to design the call.**

---
## ✅ What you just built (and own)

- A working notebook that calls a **production LLM API** — on a free account
- A **measured** demonstration that prompt design changes cost *and* reliability
- A real **extraction app** that returns structured JSON your code can consume
- The instinct to **read the token meter** on every call

**Keep this notebook.** It runs forever on the free tier. Extend it tonight.

### Free tools you can keep using
Google AI Studio · Google Colab · Gemini API (free tier) · and when you're ready: Groq, HuggingFace, n8n.

---
*The ₹0 AI Engineer — Building Real Things Before You Graduate*